# 1.0 Import Libraries & Data

In [1]:
import pandas as pd
from extra.utils import load_config

In [2]:
cfg = load_config()

Config loaded successfully!

data:
  raw_path: ../data/raw/smart_manufacturing_data.csv
  interim_path: ../data/interim/cleaned.csv
  processed_path: ../data/processed/features.csv
  target_column: maintenance_required
  temporal_target_column: time_to_failure_minutes
  anomaly_target_column: failure_type
cleaning: null
features:
  rolling_window_size: 5
  sensor_columns:
  - temperature
  - vibration
  - humidity
  - pressure
  - energy_consumption
model:
  n_jobs: 2
  session_id: 42
  train_size: 0.8
  ignore_features:
  - failure_type
  - downtime_risk
  - anomaly_flag
  - machine_status
  - predicted_remaining_life
  - time_to_failure_minutes
  temporal_ignore_features:
  - failure_type
  - downtime_risk
  - anomaly_flag
  - machine_status
  - predicted_remaining_life
  - maintenance_required
  anomaly_models:
  - iforest
  - knn
  - lof
  - pca
  anomaly_ignore_features:
  - failure_type
  - downtime_risk
  - anomaly_flag
  - machine_status
  - predicted_remaining_life
  - time_to

In [3]:
df = pd.read_csv(cfg.data.raw_path)

# 2.0 Data Cleaning

In [4]:
# I know it showed 0 but I will just remove nulls and duplicates just in case.
print(f"Original shape: {df.shape}")
df = df.dropna()
print(f"After removing nulls: {df.shape}")
df = df.drop_duplicates()
print(f"After removing duplicates: {df.shape}")

Original shape: (100000, 13)
After removing nulls: (100000, 13)
After removing duplicates: (100000, 13)


In [5]:
# I will not remove the negative vibrations because it as some correlation with
df[df["vibration"] < 0]["maintenance_required"].value_counts()

maintenance_required
0    31
1     6
Name: count, dtype: int64

In [6]:
# Convert timestamp to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Convert failure_type to categorical (Although this will be dropped for now)
df["failure_type"] = df["failure_type"].astype("category")

In [7]:
df.dtypes

timestamp                   datetime64[ns]
machine_id                           int64
temperature                        float64
vibration                          float64
humidity                           float64
pressure                           float64
energy_consumption                 float64
machine_status                       int64
anomaly_flag                         int64
predicted_remaining_life             int64
failure_type                      category
downtime_risk                      float64
maintenance_required                 int64
dtype: object

In [8]:
# Not dropping here because you can drop later in pycaret's setup()
# df = df.drop(columns=cfg.model.ignore_features)

In [9]:
# Get the last date (without the time)
print(df["timestamp"].max())
last_day = df["timestamp"].dt.normalize().max()
df = df[df["timestamp"].dt.normalize() < last_day]
print(df["timestamp"].max())

2025-03-11 10:39:00
2025-03-10 23:59:00


In [10]:
# For each machine, find time until next failure event
df = df.sort_values('timestamp').reset_index(drop=True)

events = df.loc[df[cfg.data.target_column] == 1, ['machine_id', 'timestamp']] \
           .rename(columns={'timestamp': 'event_time'}) \
           .sort_values('event_time')

df = pd.merge_asof(df, events, left_on='timestamp', right_on='event_time', by='machine_id', direction='forward')

df['time_to_failure_minutes'] = (df['event_time'] - df['timestamp']).dt.total_seconds() / 60
df = df.dropna(subset=['time_to_failure_minutes']).drop(columns=['event_time'])

In [11]:
df = df.drop(columns=['event_time', 'time_to_failure'], errors='ignore')

In [12]:
df

,timestamp,machine_id,temperature,vibration,humidity,pressure,energy_consumption,machine_status,anomaly_flag,predicted_remaining_life,failure_type,downtime_risk,maintenance_required,time_to_failure_minutes
0,2025-01-01 00:00:00,39,78.61,28.65,79.96,3.73,2.16,1,0,106,Normal,0.0,0,152.0
1,2025-01-01 00:01:00,29,68.19,57.28,35.94,3.64,0.69,1,0,320,Normal,0.0,0,93.0
2,2025-01-01 00:02:00,15,98.94,50.20,72.06,1.00,2.49,1,1,19,Normal,1.0,1,0.0
3,2025-01-01 00:03:00,43,90.91,37.65,30.34,3.15,4.96,1,1,10,Normal,1.0,1,0.0
4,2025-01-01 00:04:00,8,72.32,40.69,56.71,2.68,0.63,2,0,65,Vibration Issue,0.0,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99352,2025-03-10 23:52:00,1,65.88,55.17,39.00,2.32,4.73,2,0,73,Overheating,0.0,1,0.0
99356,2025-03-10 23:56:00,2,82.56,40.32,44.85,4.98,2.95,1,0,41,Normal,0.0,0,2.0
99357,2025-03-10 23:57:00,2,68.59,45.94,38.21,1.81,4.53,1,0,463,Normal,0.0,0,1.0
99358,2025-03-10 23:58:00,2,92.13,36.37,47.24,1.62,2.01,1,1,5,Normal,1.0,1,0.0


In [13]:
df.columns

Index(['timestamp', 'machine_id', 'temperature', 'vibration', 'humidity',
       'pressure', 'energy_consumption', 'machine_status', 'anomaly_flag',
       'predicted_remaining_life', 'failure_type', 'downtime_risk',
       'maintenance_required', 'time_to_failure_minutes'],
      dtype='object')

In [14]:
df.to_csv(cfg.data.interim_path, index=False)

In [17]:
target_columns = {
    "classification": "maintenance_required",
    "regression": "time_to_failure_minutes",
    "anomaly": "failure_type",
}

sample_classification = df.drop(columns=[target_columns["classification"]]).sample(n=20, random_state=42)
sample_classification.to_csv("../data/sample/sample_classification.csv", index=False)

sample_regression = df.drop(columns=[target_columns["regression"]]).sample(n=20, random_state=42)
sample_regression.to_csv("../data/sample/sample_regression.csv", index=False)

sample_anomaly = df.drop(columns=[target_columns["anomaly"]]).sample(n=20, random_state=42)
sample_anomaly.to_csv("../data/sample/sample_anomaly.csv", index=False)